# Backend API Tester — MTS AI Hub
Все эндпоинты + замер времени ответа. Бэкенд: `http://localhost:8000`

In [5]:
import httpx, json, struct, io, time
from pathlib import Path

BASE = "http://localhost:8000"
USER_ID = "test_user"
TIMEOUT = 300  # 5 минут для тяжёлых моделей

def ok(label, resp, elapsed=None):
    icon = '✅' if resp.status_code < 400 else '❌'
    t = f' ({elapsed:.1f}s)' if elapsed else ''
    print(f"{icon} [{resp.status_code}] {label}{t}")
    try: print(json.dumps(resp.json(), ensure_ascii=False, indent=2)[:600])
    except: print(resp.text[:400])
    print()

def timed_get(url, **kw):
    t0 = time.time()
    r = httpx.get(url, timeout=TIMEOUT, **kw)
    return r, time.time() - t0

def timed_post(url, method="POST", **kw):
    t0 = time.time()
    r = getattr(httpx, method.lower())(url, timeout=TIMEOUT, **kw)
    return r, time.time() - t0

# Создаём тестовые файлы
Path('/tmp/test_doc.txt').write_text('Это тестовый документ. Python — мощный язык. FastAPI быстрый.', encoding='utf-8')

from docx import Document
doc = Document()
doc.add_heading('Тестовый документ', level=1)
doc.add_paragraph('Python — мощный язык. MTS AI Hub — платформа для ИИ.')
doc.save('/tmp/test_doc.docx')

pdf = b'%PDF-1.4\n1 0 obj<</Type/Catalog/Pages 2 0 R>>endobj\n2 0 obj<</Type/Pages/Kids[3 0 R]/Count 1>>endobj\n3 0 obj<</Type/Page/Parent 2 0 R/MediaBox[0 0 612 792]/Contents 4 0 R/Resources<</Font<</F1 5 0 R>>>>>>endobj\n4 0 obj<</Length 44>>stream\nBT /F1 12 Tf 72 700 Td (Test PDF) Tj ET\nendstream\nendobj\n5 0 obj<</Type/Font/Subtype/Type1/BaseFont/Helvetica>>endobj\nxref\n0 6\n0000000000 65535 f \ntrailer<</Size 6/Root 1 0 R>>\nstartxref\n0\n%%EOF'
Path('/tmp/test_doc.pdf').write_bytes(pdf)

sr = 16000
wav_data = b'\x00\x00' * sr
wav = io.BytesIO()
wav.write(b'RIFF')
wav.write(struct.pack('<I', 36 + len(wav_data)))
wav.write(b'WAVEfmt ')
wav.write(struct.pack('<IHHIIHH', 16, 1, 1, sr, sr*2, 2, 16))
wav.write(b'data')
wav.write(struct.pack('<I', len(wav_data)))
wav.write(wav_data)
Path('/tmp/test_audio.wav').write_bytes(wav.getvalue())

print('✅ Файлы: test_doc.txt, test_doc.pdf, test_doc.docx, test_audio.wav')

✅ Файлы: test_doc.txt, test_doc.pdf, test_doc.docx, test_audio.wav


## 1. Health Check

In [3]:
r, t = timed_get(f"{BASE}/v1/health")
ok("Health Check", r, t)

✅ [200] Health Check (3.0s)
{
  "status": "ok",
  "services": {
    "postgres": true,
    "mws_api": true,
    "image_gen": "media-service (SD)",
    "asr": "mws(whisper-turbo-local)"
  }
}



## 2. Chat — Text (mws-gpt-alpha)

In [4]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "mws-gpt-alpha",
    "messages": [{"role": "user", "content": "Привет! Кто ты?"}],
    "stream": False, "user": USER_ID
})
ok("Chat / Text (mws-gpt-alpha)", r, t)

✅ [200] Chat / Text (mws-gpt-alpha) (6.5s)
{
  "id": "chatcmpl-a7bbfe9b1e6eeed6",
  "created": 1776005670,
  "model": "mws-gpt-alpha",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Привет! Я - MWS GPT, большая языковая модель, основанная на архитектуре GPT. Я создана для помощи людям в поиске информации, ответах на вопросы и выполнении различных задач. Моя основная цель - предоставлять точные и полезные ответы на ваши вопросы. Я могу объяснять сложные понятия, давать советы и рекомендации, а также просто разговаривать на разные темы. Я здесь, 



## 3. Chat — Code (qwen3-coder-480b-a35b)

In [5]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "qwen3-coder-480b-a35b",
    "messages": [{"role": "user", "content": "Напиши функцию сортировки пузырьком на Python"}],
    "stream": False, "user": USER_ID
})
ok("Chat / Code (qwen3-coder-480b-a35b)", r, t)

✅ [200] Chat / Code (qwen3-coder-480b-a35b) (15.9s)
{
  "id": "chatcmpl-3603d74f498547d3803a91fba218d12f",
  "created": 1776005683,
  "model": "qwen3-coder-480b-a35b",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Вот пример функции сортировки пузырьком на Python:\n\n```python\ndef bubble_sort(arr):\n    n = len(arr)\n    # Проходим по всем элементам массива\n    for i in range(n):\n        # Флаг для оптимизации: если не было обменов, значит массив отсортирован\n        swapped = False\n        # Последние i элементов уже на своих местах\n        for 



## 4. Chat — Long Context (qwen2.5-72b-instruct)

In [ ]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "qwen2.5-72b-instruct",
    "messages": [{"role": "user", "content": "Объясни трансформерные архитектуры кратко"}],
    "stream": False, "user": USER_ID
})
ok("Chat / Long Context (qwen2.5-72b-instruct)", r, t)

✅ [200] Chat / Long Context (qwen2.5-72b-instruct) (20.5s)
{
  "id": "chatcmpl-24abee21197a46118f72cb9bf14f40ca",
  "created": 1776005722,
  "model": "qwen2.5-72b-instruct",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Трансформеры — это тип нейронных сетей, который был впервые предложен в 2017 году, и с тех пор стал доминировать в современных задачах обработки естественного языка (NLP).\n\nСтандартная архитектура трансформера основывается на принципе self-attention (само-внимания). Это означает что каждый элемент последовательности (слово в предложении, нап



## 5. Chat — Auto Router (model=auto, роутинг через MWS)

In [ ]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "auto",
    "messages": [{"role": "user", "content": "Как исправить ошибку 'IndexError' в Python?"}],
    "stream": False, "user": USER_ID
})
ok("Chat / Auto Router → код", r, t)

✅ [200] Chat / Auto Router → код (12.7s)
{
  "id": "chatcmpl-85e93814d7cde846",
  "created": 1776015005,
  "model": "mws-gpt-alpha",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Квантовая запутанность - фундаментальный явление в квантовой механике, которое описывает особое взаимодействие между частицами, называемое запутанностью.\n\nКогда две или более частицы запутаны, их свойства, такие как энергия, импульс или поляризация, связаны между собой в такой sposób, что изменение состояния одной частицы немедленно влияет на состояние другой част



## 6. Chat — Streaming (SSE)

In [13]:
print("🔄 Streaming:")
t0 = time.time()
chunks = []
with httpx.stream("POST", f"{BASE}/v1/chat/completions", json={
    "model": "mws-gpt-alpha",
    "messages": [{"role": "user", "content": "Расскажи короткий анекдот"}],
    "stream": True, "user": USER_ID
}, timeout=TIMEOUT) as resp:
    print(f"Status: {resp.status_code}")
    for line in resp.iter_lines():
        if line.startswith("data: ") and line != "data: [DONE]":
            try:
                delta = json.loads(line[6:])["choices"][0]["delta"].get("content", "")
                if delta: chunks.append(delta); print(delta, end="", flush=True)
            except: pass
t = time.time() - t0
print(f"\n\n✅ chunks: {len(chunks)}, time: {t:.1f}s")

🔄 Streaming:
Status: 200
Почему компьютер пошел в психолога?

Потому что у него было заблокировано все приложения! (смеёмся)

✅ chunks: 35, time: 4.3s


## 7. Embeddings (bge-m3)

In [14]:
r, t = timed_post(f"{BASE}/v1/embeddings", json={"model": "bge-m3", "input": "Тестовый текст"})
ok("Embeddings (bge-m3)", r, t)

✅ [200] Embeddings (bge-m3) (3.0s)
{
  "model": "/models/bge-m3",
  "data": [
    {
      "embedding": [
        -0.038339663,
        0.04211078,
        -0.0064238342,
        -0.0049958024,
        0.0042633,
        -0.040040363,
        0.010305864,
        0.025565937,
        -0.0187539,
        0.0048525366,
        -0.00010217767,
        -0.0018277889,
        -0.022552742,
        -0.02501136,
        -0.00080413464,
        -0.030464688,
        -0.0029623583,
        -0.00887321,
        -0.036934737,
        -0.019909265,
        -0.038413607,
        -0.0103890505,
        0.024900446,
        0.034993723,
      



## 8. List Models

In [15]:
r, t = timed_get(f"{BASE}/v1/models")
ok("List Models", r, t)

✅ [200] List Models (2.9s)
{
  "data": [
    {
      "id": "deepseek-r1-distill-qwen-32b",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "gpt-oss-20b",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "cotype-pro-vl-32b",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "llama-3.1-8b-instruct",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "QwQ-32B",
      "object": "model",
      "created"



## 9. Deep Research (SSE)

In [16]:
print("🔄 Deep Research:")
t0 = time.time()
with httpx.stream("POST", f"{BASE}/v1/research", json={"query": "Квантовые вычисления", "user_id": USER_ID}, timeout=TIMEOUT) as resp:
    print(f"Status: {resp.status_code}")
    for line in resp.iter_lines():
        if line: print(line[:120])
t = time.time() - t0
print(f"\n✅ Research done ({t:.1f}s)")

🔄 Deep Research:
Status: 200
event: progress
data: {"step": 1, "message": "Генерирую подзапросы…"}
event: progress
data: {"step": 2, "sub_queries": ["квантовые вычисления и их применение", "квантовые алгоритмы и их эффективность", "ква
event: progress
data: {"step": 3, "message": "Ищу и парсю страницы…"}
event: progress
data: {"step": 4, "pages_fetched": 14}
event: progress
data: {"step": 5, "message": "Синтезирую ответ…"}
event: done
data: {"answer": "Исходники подтвердили, что квантовые вычисления - это область исследований в области информатики, кото

✅ Research done (56.8s)


## 10. Web Search

In [ ]:
r, t = timed_post(f"{BASE}/v1/tools/web-search", json={"query": "FastAPI Python 2025", "max_results": 3})
ok("Web Search", r, t)

✅ [200] Web Search (4.5s)
{
  "results": [
    {
      "title": "fastapi · PyPI",
      "url": "https://pypi.org/project/fastapi/",
      "snippet": "Apr 1, 2026 ·\" If anyone is looking to build a productionPythonAPI, I would highly recommendFastAPI. It is beautifully designed, simple to use and highly scalable, it has become a key component in ourAPIfirst development strategy and is driving many automations and services such as our Virtual TAC Engineer."
    },
    {
      "title": "FastAPI in 2025: The Modern Python Framework ... - Medium",
      "url": "https://medium.com/pythoneers/fastapi-in-2025-the-modern-pytho



## 11. Web Parse

In [ ]:
r, t = timed_post(f"{BASE}/v1/tools/web-parse", json={"url": "https://ru.wikipedia.org/wiki/%D0%92%D0%B8%D0%BA%D0%B8%D0%BF%D0%B5%D0%B4%D0%B8%D1%8F"})
ok("Web Parse", r, t)

✅ [200] Web Parse (1.2s)
{
  "url": "https://ru.wikipedia.org/wiki/%D0%92%D0%B8%D0%BA%D0%B8%D0%BF%D0%B5%D0%B4%D0%B8%D1%8F",
  "text": "Википедия — Википедия\nПерейти к содержанию\nЭта страница защищена от редактирования (частичная защита (до автоподтверждённых)).\nМатериал из Википедии — свободной энциклопедии\nСтабильная версия\n, проверенная 30 марта 2026.\nУ этого термина существуют и другие значения, см.\nВикипедия (значения)\n.\nЭта статья — о многоязычном веб-сайте, являющемся одним из проектов\nWikimedia Foundation\n. О разделах энциклопедии на разных языках см.\nЯзыковые разделы Википедии\n; о данном (русскояз



## 12. File Upload — TXT

In [ ]:
with open('/tmp/test_doc.txt', 'rb') as f:
    r, t = timed_post(f"{BASE}/v1/files/upload",
        files={"file": ("test_doc.txt", f, "text/plain")},
        data={"user_id": USER_ID})
ok("File Upload (TXT)", r, t)

✅ [200] File Upload (TXT) (1.1s)
{
  "file_id": "60fb95ac-daa2-44f2-8b1b-a5a43a1351a0",
  "chunks": 1
}



## 13. File Upload — PDF

In [ ]:
with open('/tmp/test_doc.pdf', 'rb') as f:
    r, t = timed_post(f"{BASE}/v1/files/upload",
        files={"file": ("test_doc.pdf", f, "application/pdf")},
        data={"user_id": USER_ID})
ok("File Upload (PDF)", r, t)

✅ [200] File Upload (PDF) (0.9s)
{
  "file_id": "9172a716-c236-46c1-82ca-9e37d9f365c9",
  "chunks": 1
}



## 14. File Upload — DOCX

In [ ]:
with open('/tmp/test_doc.docx', 'rb') as f:
    r, t = timed_post(f"{BASE}/v1/files/upload",
        files={"file": ("test_doc.docx", f, "application/vnd.openxmlformats-officedocument.wordprocessingml.document")},
        data={"user_id": USER_ID})
ok("File Upload (DOCX)", r, t)

✅ [200] File Upload (DOCX) (1.6s)
{
  "file_id": "a6ca9667-bf90-441e-9529-62f720af9905",
  "chunks": 1
}



## 15. File QA (chat с файлом)

In [ ]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "auto",
    "messages": [{"role": "user", "content": "Что написано в моём документе?"}],
    "stream": False, "user": USER_ID,
    "attachments": [{"name": "test_doc.txt", "mime": "text/plain"}]
})
ok("Chat / File QA", r, t)

✅ [200] Chat / File QA (4.5s)
{
  "id": "chatcmpl-93837a7a22891672",
  "created": 1775980522,
  "model": "mws-gpt-alpha",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Извините, но я не могу получить доступ к вашему документу, поскольку я не имею возможности видеть или считывать содержимое вашего устройства. Если вы хотите узнать содержимое документа, вы можете описать его мне или предоставить необходимую информацию. \n\nЕсли вы хотите узнать содержимое документа, вы можете ответить на следующие вопросы:\n\n- Какой тип документа у



## 16. Memory — GET

In [ ]:
r, t = timed_get(f"{BASE}/v1/memory/{USER_ID}")
ok("Memory GET", r, t)

✅ [200] Memory GET (0.0s)
{
  "user_id": "test_user",
  "memories": []
}



## 17. Memory — Search

In [ ]:
r, t = timed_post(f"{BASE}/v1/memory/{USER_ID}/search", json={"query": "Python", "top_k": 5})
ok("Memory Search", r, t)

✅ [200] Memory Search (0.7s)
{
  "user_id": "test_user",
  "results": []
}



## 18. Image Generation (qwen-image через MWS)

In [22]:
r, t = timed_post(f"{BASE}/v1/image/generate", json={
    "prompt": "красивый закат над морем"
})
ok("Image Generate (qwen-image → MWS)", r, t)

✅ [200] Image Generate (qwen-image → MWS) (15.2s)
{
  "created": 1776010151,
  "background": null,
  "data": [
    {
      "b64_json": null,
      "revised_prompt": "sunset",
      "url": "https://imagegen.gpt.mws.ru/files/651888602b76e60a28f43b617a7b3c3ad9b9dd1a4531a501c55de9adff141f3/94247a4c-33f5-4b50-8628-a000e7eb8fb1-0.png"
    }
  ],
  "output_format": "png",
  "quality": "standard",
  "size": "1024x1024",
  "usage": {
    "total_tokens": 767,
    "input_tokens": 2,
    "input_tokens_details": {
      "image_tokens": 0,
      "text_tokens": 2
    },
    "output_tokens": 765
  }
}



## 19. VLM Analyze (JSON — image_url + вопрос)

In [18]:
IMG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"

r, t = timed_post(f"{BASE}/v1/vlm/analyze", json={
    "image_url": IMG_URL,
    "question": "Что изображено?",
    "model": "qwen2.5-vl"
})
ok("VLM Analyze", r, t)

✅ [200] VLM Analyze (22.2s)
{
  "answer": null,
  "fallback": true,
  "message": "Server error '500 Internal Server Error' for url 'https://api.gpt.mws.ru/v1/chat/completions'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500"
}



## 20. PPTX Generation

In [ ]:
t0 = time.time()
r = httpx.post(f"{BASE}/v1/tools/generate-pptx", json={
    "topic": "Искусственный интеллект",
    "slide_count": 5, "language": "ru"
}, timeout=TIMEOUT)
t = time.time() - t0
if r.status_code == 200 and 'application' in r.headers.get('content-type', ''):
    out = Path('/tmp/test_output.pptx')
    out.write_bytes(r.content)
    print(f"✅ [200] PPTX saved → {out} ({len(r.content):,} bytes) ({t:.1f}s)")
else:
    ok("PPTX Generate", r, t)

❌ [500] PPTX Generate (3.5s)
Internal Server Error



## 21. Voice Message (whisper-turbo-local через MWS)

In [ ]:
with open('/tmp/test_audio.wav', 'rb') as f:
    t0 = time.time()
    r = httpx.post(f"{BASE}/v1/voice/message",
        files={"audio": ("test_audio.wav", f, "audio/wav")},
        data={"user_id": USER_ID},
        timeout=TIMEOUT)
    t = time.time() - t0
if r.status_code == 200:
    print(f"✅ [200] Voice pipeline OK ({t:.1f}s)")
    print(f"  Transcript: {r.headers.get('X-Transcript', '?')}")
    print(f"  Answer: {r.headers.get('X-Answer', '?')[:200]}")
    print(f"  Audio: {len(r.content):,} bytes")
elif r.status_code == 503:
    print(f"⚠️  [503] ASR недоступен ({t:.1f}s) — whisper-turbo-local и media-service оба не ответили")
else:
    ok("Voice Message", r, t)

❌ [500] Voice Message (4.6s)
Internal Server Error



## 22. Router Detection — все типы задач (через MWS API)

In [ ]:
cases = [
    ("Привет, как дела?",                          "text",          "mws-gpt-alpha"),
    ("Напиши функцию сортировки на Python",         "code",          "qwen3-coder-480b-a35b"),
    ("Исследуй тему квантовых вычислений",          "deep_research", "qwen2.5-72b-instruct"),
    ("Зайди на https://example.com и расскажи",     "web_parse",     "mws-gpt-alpha"),
    ("Нарисуй закат над морем",                     "image_gen",     "qwen-image"),
    ("Глубокий анализ блокчейна",                  "deep_research", "qwen2.5-72b-instruct"),
    ("Реализуй алгоритм Дейкстры на Go",           "code",          "qwen3-coder-480b-a35b"),
]
print(f"{'Ожид. тип':<18} {'Ожид. модель':<30} {'Время':>7} Статус | Сообщение")
print("-" * 95)
for msg, exp_task, exp_model in cases:
    try:
        t0 = time.time()
        r = httpx.post(f"{BASE}/v1/chat/completions", json={
            "model": "auto",
            "messages": [{"role": "user", "content": msg}],
            "stream": False, "user": USER_ID
        }, timeout=TIMEOUT)
        t = time.time() - t0
        icon = '✅' if r.status_code < 400 else '❌'
        print(f"{exp_task:<18} {exp_model:<30} {t:>6.1f}s {icon} [{r.status_code}] | {msg[:40]}")
    except Exception as e:
        print(f"{exp_task:<18} {exp_model:<30}         ❌ ERR: {e}")

Ожид. тип          Ожид. модель                     Время Статус | Сообщение
-----------------------------------------------------------------------------------------------
text               mws-gpt-alpha                    18.1s ✅ [200] | Объясни, что такое квантовая запутанност
code               qwen3-coder-480b-a35b             9.6s ✅ [200] | Как исправить ошибку 'IndexError' в Pyth
deep_research      qwen2.5-72b-instruct             24.6s ✅ [200] | Напиши пример функции на Python, которая
web_parse          mws-gpt-alpha                    16.0s ✅ [200] | Зайди на https://example.com и расскажи
image_gen          qwen-image                       27.0s ✅ [200] | Сгенерируй картину леса в стиле Ван Гога
deep_research      qwen2.5-72b-instruct             16.3s ✅ [200] | Реши задачу: найти интеграл от x^2 * e^x


## 23. Роутер — детерминированные правила (без API)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('MWS_API_KEY', 'test')
os.environ.setdefault('DATABASE_URL', 'postgresql+asyncpg://u:p@localhost/db')
os.environ.setdefault('REDIS_URL', 'redis://localhost:6379/0')

from app.config import get_settings
from app.services.router_client import RouterClient

router = RouterClient(get_settings())

cases = [
    ('Привет, как дела?',                        [],                                          None),
    ('Напиши функцию сортировки на Python',       [],                                          'code'),
    ('Реализуй алгоритм Дейкстры',               [],                                          'code'),
    ('Исследуй тему квантовых вычислений',        [],                                          'deep_research'),
    ('Глубокий анализ блокчейна',                [],                                          'deep_research'),
    ('Зайди на https://example.com',              [],                                          'web_parse'),
    ('',                                          [{'name':'voice.mp3','mime':'audio/mpeg'}],  'asr'),
    ('что на фото?',                              [{'name':'img.jpg','mime':'image/jpeg'}],    'vlm'),
    ('разбери документ',                          [{'name':'doc.pdf','mime':'application/pdf'}],'file_qa'),
    ('',                                          [{'name':'report.docx','mime':'application/vnd.openxmlformats-officedocument.wordprocessingml.document'}], 'file_qa'),
]

print(f"{'Сообщение':<45} {'Ожидается':<15} {'Результат':<15} OK?")
print('-' * 80)
passed = 0
for msg, att, expected in cases:
    result = router._deterministic(msg, att)
    got = result.task_type if result else None
    match = got == expected
    if match: passed += 1
    label = msg or att[0]['name']
    print(f"{label[:44]:<45} {str(expected):<15} {str(got):<15} {'✅' if match else '❌'}")
print(f'\nДетерминированный роутер: {passed}/{len(cases)} ✅')

Сообщение                                     Ожидается       Результат       OK?
--------------------------------------------------------------------------------
Привет, как дела?                             None            None            ✅
Напиши функцию сортировки на Python           code            code            ✅
Реализуй алгоритм Дейкстры                    code            code            ✅
Исследуй тему квантовых вычислений            deep_research   deep_research   ✅
Глубокий анализ блокчейна                     deep_research   deep_research   ✅
Зайди на https://example.com                  web_parse       web_parse       ✅
voice.mp3                                     asr             asr             ✅
что на фото?                                  vlm             vlm             ✅
разбери документ                              file_qa         file_qa         ✅
report.docx                                   file_qa         file_qa         ✅

Детерминированный роутер: 10/10 ✅


## 24. Summary — All Endpoints

## 25. История чата — сохранение и получение

In [ ]:
CONV_ID = "test-conv-" + USER_ID

# 25.1 Отправляем сообщение с conversation_id
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "auto",
    "messages": [{"role": "user", "content": "Привет! Я АНдрей?"}],
    "stream": False,
    "user": USER_ID,
    "conversation_id": CONV_ID,
})
ok("Chat with conversation_id", r, t)
time.sleep(0.5)  # дать время fire-and-forget сохранению


TypeError: can only concatenate str (not "int") to str

In [32]:
# 25.2 Список диалогов пользователя
r, t = timed_get(f"{BASE}/v1/history/{USER_ID}")
ok("List conversations", r, t)
if r.status_code == 200:
    convs = r.json().get("conversations", [])
    print(f"  Найдено диалогов: {len(convs)}")
    for c in convs[:3]:
        print(f"  - {c['id']} | {c['title']!r} | {c['updated_at']}")


✅ [200] List conversations (2.2s)
{
  "conversations": [
    {
      "id": "test-conv-test_user",
      "title": "Расскажи о планете Марс кратко",
      "created_at": "2026-04-12T16:30:22.746068",
      "updated_at": "2026-04-12T16:30:22.757504"
    }
  ]
}

  Найдено диалогов: 1
  - test-conv-test_user | 'Расскажи о планете Марс кратко' | 2026-04-12T16:30:22.757504


In [31]:
# 25.3 Сообщения конкретного диалога
r, t = timed_get(f"{BASE}/v1/history/{USER_ID}/{CONV_ID}")
ok("Get conversation messages", r, t)
if r.status_code == 200:
    msgs = r.json().get("messages", [])
    print(f"  Сообщений: {len(msgs)}")
    for m in msgs:
        print(f"  [{m['role']}] {m['content'][:80]!r}")


✅ [200] Get conversation messages (2.3s)
{
  "conversation_id": "test-conv-test_user",
  "messages": [
    {
      "id": "3f22ed79-846d-4794-b8ce-ad7109932cba",
      "role": "user",
      "content": "Расскажи о планете Марс кратко",
      "token_count": null,
      "created_at": "2026-04-12T16:30:22.757504"
    },
    {
      "id": "27ebccd6-0947-4c0c-bb93-62afad648d45",
      "role": "assistant",
      "content": "Марс — четвертая по отдаленности от Солнца планета в Солнечной системе. Это планета красного цвета с тонами желтого и коричневого. Марс — это планета с карбонатными скалами, известными как «оксиды железа и магния». Планет

  Сообщений: 4
  [user] 'Расскажи о планете Марс кратко'
  [assistant] 'Марс — четвертая по отдаленности от Солнца планета в Солнечной системе. Это план'
  [user] 'Привет! Я АНдрей?'
  [assistant] 'Привет, Андрей! Приятно познакомиться! Как я могу помочь тебе сегодня?'


In [ ]:
# 25.4 Переименование диалога
r, t = timed_post(f"{BASE}/v1/history/{USER_ID}/{CONV_ID}", method="PATCH",
    json={"title": "Тест — Марс"})
ok("Rename conversation", r, t)


In [ ]:
# 25.5 Второе сообщение — проверяем что история накапливается
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "auto",
    "messages": [{"role": "user", "content": "А Юпитер?"}],
    "stream": False,
    "user": USER_ID,
    "conversation_id": CONV_ID,
})
ok("Second message in conversation", r, t)
time.sleep(0.5)

r2, t2 = timed_get(f"{BASE}/v1/history/{USER_ID}/{CONV_ID}")
if r2.status_code == 200:
    msgs = r2.json().get("messages", [])
    print(f"  Теперь сообщений: {len(msgs)} (ожидаем 4: user+assistant x2)")


In [ ]:
# 25.6 Удаление диалога
r, t = timed_post(f"{BASE}/v1/history/{USER_ID}/{CONV_ID}", method="DELETE")
ok("Delete conversation", r, t)

# Проверяем что удалился
r2, _ = timed_get(f"{BASE}/v1/history/{USER_ID}/{CONV_ID}")
print(f"  После удаления: {r2.status_code} (ожидаем 404)")


## 26. Память — полный цикл

In [34]:
# 26.1 Сохранение памяти через чат
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "auto",
    "messages": [{"role": "user", "content": "Запомни: меня зовут Даниил и я люблю Python"}],
    "stream": False,
    "user": USER_ID,
})
ok("Save memory via chat", r, t)


✅ [200] Save memory via chat (6.0s)
{
  "id": "chatcmpl-9771b1a5337fdee8",
  "created": 1776011659,
  "model": "mws-gpt-alpha",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Приятно познакомиться, Даниил! Я запомнил, что ты любишь Python. Если у тебя есть какие-либо вопросы или проблемы по этому языку программирования, я всегда готов помочь. Следуй за мной, и мы обязательно найдем решения для всех твоих интересов!",
        "role": "assistant"
      }
    }
  ],
  "usage": {
    "completion_tokens": 69,
    "prompt_tokens": 306,
    "to



In [35]:
# 26.2 Получение всей памяти
r, t = timed_get(f"{BASE}/v1/memory/{USER_ID}")
ok("Get all memories", r, t)
if r.status_code == 200:
    mems = r.json().get("memories", r.json())
    if isinstance(mems, list):
        print(f"  Воспоминаний: {len(mems)}")
        for m in mems[:3]:
            txt = m.get("text","") if isinstance(m,dict) else str(m)
            print(f"  - {txt[:80]!r}")


✅ [200] Get all memories (2.3s)
{
  "user_id": "test_user",
  "memories": []
}

  Воспоминаний: 0


In [ ]:
# 26.3 Семантический поиск по памяти
r, t = timed_post(f"{BASE}/v1/memory/{USER_ID}/search", json={
    "query": "имя пользователя",
    "top_k": 5
})
ok("Memory search", r, t)
if r.status_code == 200:
    results = r.json().get("results", [])
    print(f"  Найдено: {len(results)}")
    for res in results[:3]:
        txt = res.get("text","") if isinstance(res,dict) else str(res)
        score = res.get("score", "") if isinstance(res,dict) else ""
        print(f"  score={score:.3f} text={txt[:60]!r}" if score else f"  {txt[:80]!r}")


## 27. Все модели MWS — проверка доступности

In [ ]:
# Список всех моделей
r, t = timed_get(f"{BASE}/v1/models")
ok("List models", r, t)
if r.status_code == 200:
    models = [m["id"] for m in r.json().get("data", [])]
    print(f"  Доступно моделей: {len(models)}")
    for m in models:
        print(f"  - {m}")


In [ ]:
# Тест чата по основным моделям
test_models = [
    ("mws-gpt-alpha",          "текст/роутинг"),
    ("qwen3-coder-480b-a35b",  "код"),
    ("qwen2.5-72b-instruct",   "аналитика"),
    ("QwQ-32B",                "reasoning"),
    ("llama-3.3-70b-instruct", "общее"),
]

for model_id, purpose in test_models:
    r, t = timed_post(f"{BASE}/v1/chat/completions", json={
        "model": model_id,
        "messages": [{"role":"user","content":"Скажи одно слово: ОК"}],
        "max_tokens": 10,
        "stream": False,
    })
    ans = ""
    if r.status_code == 200:
        try: ans = r.json()["choices"][0]["message"]["content"][:30]
        except: pass
    ok(f"{model_id} ({purpose})", r, t, ans)


In [ ]:
# Тест эмбеддингов
embed_models = ["bge-m3", "qwen3-embedding-8b", "BAAI/bge-multilingual-gemma2"]
for model_id in embed_models:
    r, t = timed_post(f"{BASE}/v1/embeddings", json={
        "model": model_id,
        "input": "Тестовый текст для эмбеддинга"
    })
    dim = ""
    if r.status_code == 200:
        try: dim = f"dim={len(r.json()['data'][0]['embedding'])}"
        except: pass
    ok(f"Embeddings {model_id}", r, t, dim)


In [ ]:
# Тест ASR
import os
voice_file = os.path.join(os.path.dirname("backend/tests/"), "backend/tests/voice_sample.mp3")
if os.path.exists(voice_file):
    with open(voice_file, "rb") as f:
        audio_bytes = f.read()
    r, t = timed_post(f"{BASE}/v1/voice/message",
        files={"audio": ("voice_sample.mp3", audio_bytes, "audio/mpeg")},
        data={"user_id": USER_ID})
    if r.status_code == 200:
        ct = r.headers.get("content-type","")
        if "audio" in ct:
            ok(f"Voice pipeline (TTS ok, {len(r.content)}B {ct})", r, t)
        else:
            d = r.json()
            ok(f"Voice pipeline (TTS fallback)", r, t, f"transcript={d.get('transcript','')[:40]!r}")
    else:
        ok("Voice pipeline", r, t)
else:
    print("⚠️  voice_sample.mp3 не найден, пропускаю")


In [ ]:
endpoints = [
    ("GET",  "/v1/health",                    None),
    ("POST", "/v1/chat/completions",          {"model":"mws-gpt-alpha","messages":[{"role":"user","content":"hi"}],"stream":False}),
    ("POST", "/v1/embeddings",                {"model":"bge-m3","input":"test"}),
    ("GET",  "/v1/models",                    None),
    ("POST", "/v1/tools/web-search",          {"query":"test","max_results":1}),
    ("POST", "/v1/tools/web-parse",           {"url":"https://example.com"}),
    ("POST", "/v1/tools/generate-pptx",       {"topic":"AI","slide_count":3,"language":"ru"}),
    ("GET",  f"/v1/memory/{USER_ID}",         None),
    ("POST", f"/v1/memory/{USER_ID}/search",  {"query":"test","top_k":3}),
    ("POST", "/v1/image/generate",            {"prompt":"cat","model":"qwen-image","size":"1024x1024"}),
    ("POST", "/v1/vlm/analyze",               {"image_url":"https://example.com/i.png","question":"test","model":"qwen2.5-vl"}),
]
print(f"{'Method':<6} {'Endpoint':<40} {'Status':>6} {'Time':>7}  Result")
print("-" * 70)
passed = 0
for method, path, body in endpoints:
    try:
        t0 = time.time()
        r = httpx.get(f"{BASE}{path}", timeout=TIMEOUT) if method == "GET" else httpx.post(f"{BASE}{path}", json=body, timeout=TIMEOUT)
        t = time.time() - t0
        icon = "✅" if r.status_code < 400 else "⚠️ "
        if r.status_code < 400: passed += 1
        print(f"{method:<6} {path:<40} {r.status_code:>6} {t:>6.1f}s  {icon}")
    except Exception as e:
        print(f"{method:<6} {path:<40} {'ERR':>6}          ❌ {str(e)[:30]}")
print(f"\n{'='*70}\nИтого: {passed}/{len(endpoints)} OK")

Method Endpoint                                 Status    Time  Result
----------------------------------------------------------------------
GET    /v1/health                                  200    0.9s  ✅
POST   /v1/chat/completions                        200    2.8s  ✅
POST   /v1/embeddings                              200    0.6s  ✅
GET    /v1/models                                  200    0.7s  ✅
POST   /v1/tools/web-search                        200    1.5s  ✅
POST   /v1/tools/web-parse                         200    0.4s  ✅
POST   /v1/tools/generate-pptx                     200    2.0s  ✅
GET    /v1/memory/test_user                        200    0.0s  ✅
POST   /v1/memory/test_user/search                 200    0.8s  ✅
POST   /v1/image/generate                          200   23.5s  ✅
POST   /v1/vlm/analyze                             200   20.1s  ✅

Итого: 11/11 OK
